# WP1A — Benchmark reproduzível de métodos clássicos de ML no Tennessee Eastman Process
**Disciplina MEI0028/MEI0015 – Modelagem e Simulação · PUC Goiás · 2026 · Prof. Clarimar J. Coelho**

Base: Rieth et al. (2017), Harvard Dataverse, DOI 10.7910/DVN/6C3JR1 (domínio público) — 21 classes, 52 variáveis, 500 execuções/classe.
Protocolo: divisão **por execução (run)**; hiperparâmetros só na validação; teste lido uma única vez; 5 sementes; F1 macro + MCC; custo computacional medido no mesmo hardware.

Estes notebooks reproduzem exatamente o código executado. Cada célula `%%writefile` grava o script numerado; a célula seguinte o executa.

## Notebook 3 de 3 — Treino final, avaliação e registro de figuras/tabelas
> **Os tempos oficiais do artigo foram medidos em um único computador (Apple M3, 8 GB, 4 núcleos, `nice 19`).** Executar aqui reproduz as métricas de acerto, não os tempos.

In [ ]:
import os; os.makedirs("/content/ProjetoA_WP1A/src",exist_ok=True); os.chdir("/content/ProjetoA_WP1A")
os.environ["WP1A_ROOT"]="/content/ProjetoA_WP1A"; os.environ["NJOBS"]="2"
!pip -q install pyreadr pyarrow tabulate 2>/dev/null; print("ambiente pronto")

### 9. Congelamento da configuração
Mescla as buscas (Mac + Colab) e escolhe, por modelo, a melhor configuração pelo F1 macro de validação. A partir daqui nada muda antes de o teste ser lido.

In [ ]:
%%writefile /content/ProjetoA_WP1A/src/06b_merge_hp.py
"""06b_merge_hp.py — mescla os resultados da busca de hiperparâmetros (Mac: 5 modelos CPU; Colab: logreg + xgboost GPU),
escolhe a melhor configuração por modelo pelo F1 macro na validação e grava configs/hiperparametros_escolhidos.json."""
import os, json
ROOT=os.environ.get("WP1A_ROOT") or os.path.join(os.path.dirname(os.path.abspath(__file__)),".."); META=os.path.join(ROOT,"results","metadata"); CFG=os.path.join(ROOT,"configs")
fontes={"mac":os.path.join(META,"busca_hp_resultados.json"),"colab":os.path.join(META,"busca_hp_resultados_COLAB.json")}
merged={}
for origem,f in fontes.items():
    if not os.path.exists(f): print("ausente:",f); continue
    for m,rs in json.load(open(f)).items():
        for r in rs:
            k=json.dumps(r["params"],sort_keys=True); merged.setdefault(m,{})
            if k not in merged[m]: merged[m][k]=dict(r,origem=origem)
out={m:list(v.values()) for m,v in merged.items()}; json.dump(out,open(os.path.join(META,"busca_hp_resultados_MERGED.json"),"w"),indent=1)
best={m:max(v,key=lambda r:r["f1_macro"]) for m,v in out.items()}; json.dump(best,open(os.path.join(CFG,"hiperparametros_escolhidos.json"),"w"),indent=2)
esperados=["regressao_logistica","arvore_decisao","random_forest","gradient_boosting","xgboost","svm_nystroem"]; falt=[m for m in esperados if m not in best]
for m in esperados:
    if m in best: print(f"{m:<22} {len(out[m]):>2} configs  melhor F1m={best[m]['f1_macro']:.4f} ({best[m]['origem']})  {best[m]['params']}")
if falt: raise SystemExit(f"FALTAM modelos na busca: {falt}")
print("MERGE HP COMPLETO")


In [ ]:
!python3 -u src/06b_merge_hp.py

### 10. Treino final e avaliação no teste (WP1A §9.6 · §10)
Cada (modelo, semente) roda em **subprocesso isolado** para medir o pico de memória por modelo. Treino: 100 runs/classe; teste: os 500 runs/classe do *Testing* (10,08 M amostras), em lotes. Registra tempo de treino, tempo de inferência, tamanho do modelo serializado e pico de memória, além das métricas com rótulo físico (primário) e rótulo do run (secundário). Sementes 42, 123, 2024, 7, 555.

In [ ]:
%%writefile /content/ProjetoA_WP1A/src/07_treino_final.py
"""07_treino_final.py — Etapa 6 do WP1A (§9.6): treino final com configuração CONGELADA e avaliação no teste.
Roda EXCLUSIVAMENTE no Mac (única fonte dos tempos oficiais — objetivo específico 5).
Cada (modelo, semente) executa em subprocesso isolado → pico de memória (ru_maxrss) por modelo.
Treino: 100 runs/classe (manifesto 'treino'). Teste: TODOS os 500 runs/classe do Testing, em lotes.
Sementes {42,123,2024,7,555} (mesmas de Koçak et al., 2026). Runs de treino fixos; varia a semente do modelo."""
import os, sys, json, time, argparse, resource, subprocess, numpy as np, pandas as pd, joblib, warnings; warnings.filterwarnings("ignore")
import pyarrow.parquet as pq
ROOT=os.environ.get("WP1A_ROOT") or os.path.join(os.path.dirname(os.path.abspath(__file__)),"..")
PROC=os.path.join(ROOT,"data","processed"); META=os.path.join(ROOT,"results","metadata"); CFG=os.path.join(ROOT,"configs")
MOD=os.path.join(ROOT,"models"); PRED=os.path.join(ROOT,"results","predictions"); OUT=os.path.join(META,"treino_final"); os.makedirs(OUT,exist_ok=True)
SEEDS=[42,123,2024,7,555]; MODELOS=["regressao_logistica","arvore_decisao","random_forest","gradient_boosting","xgboost","svm_nystroem"]
NJ=int(os.environ.get("NJOBS",4)); ID=["faultNumber","simulationRun","sample"]
ONSET_TR,ONSET_TE=20,160  # Rieth: falha introduzida em 1 h (Training) e 8 h (Testing); amostras <= onset são fisicamente normais

def build(nome,p,seed):
    from sklearn.linear_model import LogisticRegression; from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
    sys.path.insert(0,os.path.dirname(os.path.abspath(__file__))); from nystroem_sgd import NystroemSGD
    from sklearn.pipeline import make_pipeline; from sklearn.preprocessing import StandardScaler; from xgboost import XGBClassifier
    sc=StandardScaler()
    if nome=="regressao_logistica": m=LogisticRegression(max_iter=300,n_jobs=NJ,**p)
    elif nome=="arvore_decisao":    m=DecisionTreeClassifier(random_state=seed,**p)
    elif nome=="random_forest":     m=RandomForestClassifier(random_state=seed,n_jobs=NJ,**p)
    elif nome=="gradient_boosting": m=HistGradientBoostingClassifier(random_state=seed,early_stopping=False,**p)
    elif nome=="xgboost":           m=XGBClassifier(random_state=seed,tree_method="hist",device="cpu",n_jobs=NJ,subsample=0.8,colsample_bytree=0.8,**p)
    elif nome=="svm_nystroem":      m=NystroemSGD(n_components=p["n_components"],alpha=p["alpha"],epochs=5,random_state=seed,n_jobs=NJ)
    return make_pipeline(sc,m)

def load_treino(runs):
    def one(n): return pd.read_parquet(os.path.join(PROC,n+".parquet")).merge(runs[ID[:2]],on=ID[:2])
    return pd.concat([one("TEP_FaultFree_Training"),one("TEP_Faulty_Training")],ignore_index=True)

def predict_test(pipe,xcols,batch=400_000):
    """Predição em lotes sobre TODO o Testing (10,08 M linhas) — memória constante."""
    ids,preds=[],[]; t_inf=0.0; n=0
    for name in ("TEP_FaultFree_Testing","TEP_Faulty_Testing"):
        pf=pq.ParquetFile(os.path.join(PROC,name+".parquet"))
        for b in pf.iter_batches(batch_size=batch,columns=ID+xcols):
            df=b.to_pandas(); X=df[xcols].to_numpy(dtype=np.float32)
            t=time.perf_counter(); p=pipe.predict(X); t_inf+=time.perf_counter()-t; n+=len(X)
            ids.append(df[ID].to_numpy(dtype=np.int32)); preds.append(p.astype(np.int8))
    return np.vstack(ids),np.concatenate(preds),t_inf,n

def run_one(nome,seed):
    from sklearn.metrics import f1_score,matthews_corrcoef,balanced_accuracy_score,accuracy_score
    hp=json.load(open(os.path.join(CFG,"hiperparametros_escolhidos.json")))[nome]["params"]
    man=pd.read_csv(os.path.join(META,"manifesto_divisao.csv")); tr=load_treino(man[man.conjunto=="treino"])
    xcols=[c for c in tr.columns if c not in ID]; X=tr[xcols].to_numpy(dtype=np.float32)
    y=np.where(tr["sample"]<=ONSET_TR,0,tr.faultNumber).astype(np.int32); del tr  # rótulo físico
    pipe=build(nome,hp,seed); m0=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    t=time.perf_counter(); pipe.fit(X,y); t_fit=time.perf_counter()-t
    pico=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/2**20  # MB (macOS: bytes)
    del X; mp=os.path.join(MOD,f"{nome}_seed{seed}.joblib"); joblib.dump(pipe,mp,compress=3); size=os.path.getsize(mp)
    ids,p,t_inf,n=predict_test(pipe,xcols); y_run=ids[:,0]; yt=np.where(ids[:,2]<=ONSET_TE,0,y_run).astype(np.int8)
    out=pd.DataFrame(ids,columns=ID); out["y_pred"]=p; out.to_parquet(os.path.join(PRED,f"{nome}_seed{seed}.parquet"),index=False)
    r=dict(modelo=nome,seed=seed,params=hp,n_treino=int(len(y)),n_teste=int(n),tempo_treino_s=round(t_fit,2),
           tempo_inferencia_total_s=round(t_inf,2),tempo_inferencia_por_amostra_us=round(1e6*t_inf/n,3),tamanho_modelo_bytes=size,
           pico_memoria_MB=round(pico,1),acuracia=float(accuracy_score(yt,p)),acuracia_balanceada=float(balanced_accuracy_score(yt,p)),
           f1_macro=float(f1_score(yt,p,average="macro")),f1_ponderada=float(f1_score(yt,p,average="weighted")),mcc=float(matthews_corrcoef(yt,p)),
           hardware="Apple M3 8GB (4P+4E)",n_jobs=NJ)
    json.dump(r,open(os.path.join(OUT,f"{nome}_seed{seed}.json"),"w"),indent=2); print(json.dumps(r),flush=True)

if __name__=="__main__":
    ap=argparse.ArgumentParser(); ap.add_argument("--modelo"); ap.add_argument("--seed",type=int); a=ap.parse_args()
    if a.modelo: run_one(a.modelo,a.seed); sys.exit(0)
    for nome in MODELOS:
        for seed in SEEDS:
            f=os.path.join(OUT,f"{nome}_seed{seed}.json")
            if os.path.exists(f): print(f"skip {nome} seed{seed}",flush=True); continue
            print(f"[{time.strftime('%H:%M')}] {nome} seed{seed} ...",flush=True); t=time.time()
            rc=subprocess.run([sys.executable,"-u",__file__,"--modelo",nome,"--seed",str(seed)],env={**os.environ,"WP1A_ROOT":ROOT}).returncode
            print(f"   rc={rc} em {(time.time()-t)/60:.1f} min",flush=True)
    print("TREINO FINAL COMPLETO",flush=True)


In [ ]:
!NJOBS=2 python3 -u src/07_treino_final.py

### 11. Avaliação, estatística e figuras (WP1A §10 · §11 · §12)
Média ± dp e IC 95% (t, gl = 4) nas 5 sementes; matrizes de confusão agregadas; F1 por classe; pares mais confundidos; Friedman + Wilcoxon pareado com correção de Holm; métricas em nível de execução (voto majoritário, comparável a Koçak et al., 2026); gráficos de F1/MCC, custo e compromisso desempenho × custo.

In [ ]:
%%writefile /content/ProjetoA_WP1A/src/08_avaliacao.py
"""08_avaliacao.py — Etapa 6 (§9.6), Métricas (§10), Análise estatística (§11) e Insumos (§12) do WP1A.
Lê results/metadata/treino_final/*.json e results/predictions/*.parquet; gera tabelas (CSV) e figuras (PNG).
Rótulo PRIMÁRIO no teste = físico (sample<=160 → normal); secundário = rótulo do run (comparabilidade)."""
import os, glob, json, itertools, numpy as np, pandas as pd, matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, f1_score, matthews_corrcoef
ROOT=os.environ.get("WP1A_ROOT") or os.path.join(os.path.dirname(os.path.abspath(__file__)),"..")
META=os.path.join(ROOT,"results","metadata"); PRED=os.path.join(ROOT,"results","predictions"); TAB=os.path.join(ROOT,"results","tables"); FIG=os.path.join(ROOT,"results","figures"); CFG=os.path.join(ROOT,"configs")
ONSET_TE=160; CL=list(range(21)); NOME={"regressao_logistica":"Regressão Logística","arvore_decisao":"Árvore de Decisão","random_forest":"Random Forest","gradient_boosting":"Gradient Boosting","xgboost":"XGBoost","svm_nystroem":"SVM (Nyström)"}
ORD=list(NOME); SEEDS=[42,123,2024,7,555]
runs=[json.load(open(f)) for f in sorted(glob.glob(os.path.join(META,"treino_final","*.json")))]
R=pd.DataFrame(runs); R["modelo_nome"]=R.modelo.map(NOME); R.to_csv(os.path.join(TAB,"tab03b_metricas_por_seed.csv"),index=False)
modelos=[m for m in ORD if m in set(R.modelo)]; print("modelos com resultados:",modelos,"| execuções:",len(R),flush=True)
tq=stats.t.ppf(0.975,df=4)
def agg(col): 
    g=R.groupby("modelo")[col]; return pd.DataFrame({"media":g.mean(),"dp":g.std(ddof=1),"ic95":tq*g.std(ddof=1)/np.sqrt(g.count()),"n":g.count()}).reindex(modelos)
# ---- Tab 2: hiperparâmetros
hp=json.load(open(os.path.join(CFG,"hiperparametros_escolhidos.json")))
pd.DataFrame([dict(Modelo=NOME[m],Hiperparametros=json.dumps(hp[m]["params"]),F1_macro_validacao=round(hp[m]["f1_macro"],4)) for m in modelos]).to_csv(os.path.join(TAB,"tab02_hiperparametros.csv"),index=False)
# ---- Tab 3: métricas globais (média ± dp, IC95) e Tab 5: custo
MET=["acuracia","acuracia_balanceada","f1_macro","f1_ponderada","mcc","rotulo_run_acuracia","rotulo_run_f1_macro","rotulo_run_mcc"]
CUSTO=["tempo_treino_s","tempo_inferencia_total_s","tempo_inferencia_por_amostra_us","tamanho_modelo_bytes","pico_memoria_MB"]
t3=pd.concat({c:agg(c) for c in MET},axis=1); t3.index=[NOME[m] for m in t3.index]; t3.round(4).to_csv(os.path.join(TAB,"tab03_metricas_globais.csv"))
t5=pd.concat({c:agg(c) for c in CUSTO},axis=1); t5.index=[NOME[m] for m in t5.index]; t5.round(3).to_csv(os.path.join(TAB,"tab05_custo_computacional.csv"))
# ---- por classe, matrizes de confusão, run-level (lê predições)
pc_rows=[]; cms={m:np.zeros((21,21),dtype=np.int64) for m in modelos}; rl_rows=[]
for m in modelos:
    for s in SEEDS:
        f=os.path.join(PRED,f"{m}_seed{s}.parquet")
        if not os.path.exists(f): continue
        d=pd.read_parquet(f); yt=np.where(d["sample"]<=ONSET_TE,0,d.faultNumber).astype(np.int8); yp=d.y_pred.to_numpy()
        p,r,f1,sup=precision_recall_fscore_support(yt,yp,labels=CL,zero_division=0)
        for c in CL: pc_rows.append(dict(modelo=m,seed=s,classe=c,precisao=p[c],revocacao=r[c],f1=f1[c],suporte=int(sup[c])))
        cms[m]+=confusion_matrix(yt,yp,labels=CL)
        # run-level: voto majoritário por (faultNumber, simulationRun); rótulo do run
        cnt=d.groupby(["faultNumber","simulationRun","y_pred"]).size().reset_index(name="n")
        vote=cnt.sort_values("n",ascending=False).drop_duplicates(["faultNumber","simulationRun"])
        rl_rows.append(dict(modelo=m,seed=s,run_level_f1_macro=f1_score(vote.faultNumber,vote.y_pred,average="macro"),run_level_acuracia=(vote.faultNumber==vote.y_pred).mean(),run_level_mcc=matthews_corrcoef(vote.faultNumber,vote.y_pred),n_runs=len(vote)))
        del d; print(f"  {m} seed{s} ok",flush=True)
PC=pd.DataFrame(pc_rows); t4=PC.groupby(["modelo","classe"])[["precisao","revocacao","f1"]].mean().reset_index(); t4["modelo"]=t4.modelo.map(NOME)
t4.round(4).to_csv(os.path.join(TAB,"tab04_metricas_por_classe.csv"),index=False)
RL=pd.DataFrame(rl_rows); RL.to_csv(os.path.join(TAB,"tab08b_run_level_por_seed.csv"),index=False)
t8=RL.groupby("modelo")[["run_level_f1_macro","run_level_acuracia","run_level_mcc"]].agg(["mean","std"]).reindex(modelos); t8.index=[NOME[m] for m in t8.index]; t8.round(4).to_csv(os.path.join(TAB,"tab08_run_level.csv"))
# ---- Tab 6: falhas mais confundidas (pares verdadeiro→predito, fora da diagonal, normalizado por linha)
conf=[]
for m in modelos:
    cm=cms[m]; cmn=cm/np.maximum(cm.sum(1,keepdims=True),1)
    for i in CL:
        for j in CL:
            if i!=j and cm[i,j]>0: conf.append(dict(modelo=NOME[m],verdadeira=i,predita=j,fracao_da_classe=cmn[i,j],n=int(cm[i,j])))
    pd.DataFrame(cmn,index=CL,columns=CL).round(4).to_csv(os.path.join(TAB,f"tab_matriz_confusao_{m}.csv"))
CF=pd.DataFrame(conf).sort_values(["modelo","fracao_da_classe"],ascending=[True,False]); CF.groupby("modelo").head(10).round(4).to_csv(os.path.join(TAB,"tab06_falhas_confundidas.csv"),index=False)
# falhas mais difíceis: F1 médio por classe entre modelos
dif=t4.groupby("classe").f1.mean().sort_values(); dif.round(4).to_csv(os.path.join(TAB,"tab06b_dificuldade_por_classe.csv"))
# ---- Tab 7: estatística (§11) — Friedman + Wilcoxon pareado com Holm, sobre seeds
def holm(pvals):
    n=len(pvals); order=np.argsort(pvals); adj=np.empty(n)
    for rank,i in enumerate(order): adj[i]=min(1.0,pvals[i]*(n-rank))
    for k in range(1,n): adj[order[k]]=max(adj[order[k]],adj[order[k-1]])
    return adj
st=[]
for met in ["f1_macro","mcc"]:
    piv=R.pivot(index="seed",columns="modelo",values=met)[modelos].dropna()
    if len(piv)>=2 and piv.shape[1]>=3:
        fr=stats.friedmanchisquare(*[piv[m] for m in modelos]); st.append(dict(metrica=met,teste="Friedman",comparacao="todos",estatistica=fr.statistic,p=fr.pvalue,p_holm=None))
        pares=list(itertools.combinations(modelos,2)); ps=[]
        for a,b in pares:
            try: w=stats.wilcoxon(piv[a],piv[b]); ps.append(w.pvalue)
            except ValueError: ps.append(1.0)
        ph=holm(np.array(ps))
        for (a,b),p,q in zip(pares,ps,ph): st.append(dict(metrica=met,teste="Wilcoxon",comparacao=f"{NOME[a]} vs {NOME[b]}",estatistica=float(piv[a].mean()-piv[b].mean()),p=p,p_holm=q))
    piv.to_csv(os.path.join(TAB,f"tab07b_{met}_por_seed_pivot.csv"))
pd.DataFrame(st).round(5).to_csv(os.path.join(TAB,"tab07_estatistica.csv"),index=False)
# ---- Tab 9: resumo da busca de hp + trade-off do Nyström
bh=json.load(open(os.path.join(META,"busca_hp_resultados_MERGED.json") if os.path.exists(os.path.join(META,"busca_hp_resultados_MERGED.json")) else os.path.join(META,"busca_hp_resultados.json")))
pd.DataFrame([dict(Modelo=NOME.get(m,m),configs_testadas=len(v),melhor_f1_macro_val=max(r["f1_macro"] for r in v),pior_f1_macro_val=min(r["f1_macro"] for r in v)) for m,v in bh.items()]).round(4).to_csv(os.path.join(TAB,"tab09_busca_hp_resumo.csv"),index=False)
if "svm_nystroem" in bh:
    ny=pd.DataFrame([dict(**r["params"],f1_macro=r["f1_macro"],mcc=r["mcc"],tempo_s=r["tempo_treino_s"]) for r in bh["svm_nystroem"]]); ny.to_csv(os.path.join(TAB,"tab09b_nystroem_tradeoff.csv"),index=False)
    fig,ax=plt.subplots(figsize=(7,4)); ax2=ax.twinx()
    for C,g in ny.groupby("C"): g=g.sort_values("n_components"); ax.plot(g.n_components,g.f1_macro,"o-",label=f"F1 macro (C={C})"); ax2.plot(g.n_components,g.tempo_s/60,"s--",alpha=.6,label=f"tempo (C={C})")
    ax.set_xlabel("n_components (D) do Nyström"); ax.set_ylabel("F1 macro (validação)"); ax2.set_ylabel("tempo de treino (min)"); ax.set_title("SVM via Nyström: qualidade × custo em função de D"); ax.legend(loc="upper left",fontsize=8); ax2.legend(loc="lower right",fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(FIG,"fig06_svm_nystroem_tradeoff.png"),dpi=200); plt.close()
# ---- Fig 1: F1 macro e MCC com IC95
lab=[NOME[m] for m in modelos]; x=np.arange(len(modelos)); fig,ax=plt.subplots(figsize=(9,4.5))
for k,(col,off,cor) in enumerate([("f1_macro",-0.2,"#1f77b4"),("mcc",0.2,"#ff7f0e")]):
    a=agg(col); ax.bar(x+off,a.media,0.38,yerr=a.ic95,capsize=3,label={"f1_macro":"F1 macro","mcc":"MCC"}[col],color=cor)
    for xi,v in zip(x+off,a.media): ax.text(xi,v+0.01,f"{v:.3f}",ha="center",fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(lab,rotation=15); ax.set_ylim(0,1.05); ax.set_ylabel("valor no teste (rótulo físico)"); ax.set_title("F1 macro e MCC no conjunto de teste — média de 5 sementes, barras = IC 95%"); ax.legend(); plt.tight_layout(); plt.savefig(os.path.join(FIG,"fig01_f1_mcc_comparativo.png"),dpi=200); plt.close()
# ---- Fig 2: custo computacional (3 painéis, escala log)
fig,axs=plt.subplots(1,3,figsize=(13,4))
for ax,(col,tit,esc) in zip(axs,[("tempo_treino_s","Tempo de treino (s)",1),("tempo_inferencia_por_amostra_us","Inferência por amostra (µs)",1),("tamanho_modelo_bytes","Tamanho do modelo (MB)",1/2**20)]):
    a=agg(col); ax.bar(x,a.media*esc,yerr=a.ic95*esc,capsize=3,color="#2ca02c"); ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(lab,rotation=30,fontsize=8); ax.set_title(tit)
    for xi,v in zip(x,a.media*esc): ax.text(xi,v*1.15,f"{v:.3g}",ha="center",fontsize=7)
fig.suptitle("Custo computacional — mesmo hardware (Apple M3, 8 GB), média de 5 sementes"); plt.tight_layout(); plt.savefig(os.path.join(FIG,"fig02_custo_computacional.png"),dpi=200); plt.close()
# ---- Fig 3: matrizes de confusão normalizadas (agregadas nas 5 sementes) — uma por modelo
for m in modelos:
    cmn=cms[m]/np.maximum(cms[m].sum(1,keepdims=True),1); fig,ax=plt.subplots(figsize=(7.5,6.5)); im=ax.imshow(cmn,cmap="Blues",vmin=0,vmax=1)
    ax.set_xticks(CL); ax.set_yticks(CL); ax.set_xlabel("classe predita"); ax.set_ylabel("classe verdadeira (rótulo físico)"); ax.set_title(f"Matriz de confusão normalizada — {NOME[m]} (5 sementes agregadas)")
    for i in CL:
        for j in CL:
            if cmn[i,j]>=0.05: ax.text(j,i,f"{cmn[i,j]:.2f}",ha="center",va="center",fontsize=5,color="white" if cmn[i,j]>0.5 else "black")
    plt.colorbar(im,ax=ax,fraction=0.04); plt.tight_layout(); plt.savefig(os.path.join(FIG,f"fig03_matriz_confusao_{m}.png"),dpi=200); plt.close()
# ---- Fig 4: heatmap F1 por classe (modelos × classes)
H=t4.pivot(index="modelo",columns="classe",values="f1").reindex(lab); fig,ax=plt.subplots(figsize=(12,3.8)); im=ax.imshow(H.to_numpy(),cmap="viridis",vmin=0,vmax=1,aspect="auto")
ax.set_xticks(CL); ax.set_yticks(range(len(lab))); ax.set_yticklabels(lab,fontsize=8); ax.set_xlabel("classe (0 = normal; 1–20 = IDV)"); ax.set_title("F1 por classe no teste — média de 5 sementes")
for i in range(len(lab)):
    for j in CL: ax.text(j,i,f"{H.iloc[i,j]:.2f}",ha="center",va="center",fontsize=5.5,color="white" if H.iloc[i,j]<0.5 else "black")
plt.colorbar(im,ax=ax,fraction=0.02); plt.tight_layout(); plt.savefig(os.path.join(FIG,"fig04_f1_por_classe_heatmap.png"),dpi=200); plt.close()
# ---- Fig 5: trade-off F1 macro × tempo de inferência
a1=agg("f1_macro"); a2=agg("tempo_inferencia_por_amostra_us"); fig,ax=plt.subplots(figsize=(7,4.5))
for m in modelos: ax.scatter(a2.loc[m,"media"],a1.loc[m,"media"],s=80); ax.annotate(NOME[m],(a2.loc[m,"media"],a1.loc[m,"media"]),textcoords="offset points",xytext=(6,4),fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("tempo de inferência por amostra (µs, escala log)"); ax.set_ylabel("F1 macro (teste)"); ax.set_title("Compromisso entre desempenho e custo de inferência"); plt.tight_layout(); plt.savefig(os.path.join(FIG,"fig05_tradeoff_f1_vs_inferencia.png"),dpi=200); plt.close()
# ---- Fig 7: experimento do vazamento (piloto)
pv=os.path.join(META,"piloto_vazamento.json")
if os.path.exists(pv):
    v=json.load(open(pv)); ks=["acuracia","acuracia_balanceada","f1_macro","mcc"]; fig,ax=plt.subplots(figsize=(7,4)); xx=np.arange(4)
    ax.bar(xx-0.2,[v["A_por_run"][k] for k in ks],0.4,label="divisão por run (protocolo WP1A)"); ax.bar(xx+0.2,[v["B_por_amostra"][k] for k in ks],0.4,label="divisão aleatória por amostra")
    for i,k in enumerate(ks): ax.text(i-0.2,v["A_por_run"][k]+0.01,f"{v['A_por_run'][k]:.3f}",ha="center",fontsize=7); ax.text(i+0.2,v["B_por_amostra"][k]+0.01,f"{v['B_por_amostra'][k]:.3f}",ha="center",fontsize=7)
    ax.set_xticks(xx); ax.set_xticklabels(["acurácia","acur. balanceada","F1 macro","MCC"]); ax.set_ylim(0,1.1); ax.set_title("Efeito do vazamento: mesmo modelo, mesmos dados, unidade de divisão diferente"); ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(os.path.join(FIG,"fig07_vazamento.png"),dpi=200); plt.close()
# ---- metadados do ambiente (§12 item 10)
import platform, sklearn, xgboost, scipy, subprocess
json.dump(dict(python=platform.python_version(),plataforma=platform.platform(),hardware="Apple M3, 8 núcleos (4P+4E), 8 GB RAM",numpy=np.__version__,pandas=pd.__version__,scikit_learn=sklearn.__version__,xgboost=xgboost.__version__,scipy=scipy.__version__,
               sementes=SEEDS,n_jobs=int(os.environ.get("NJOBS",4)),colab_hp="Google Colab T4 (busca de hiperparâmetros apenas; tempos não usados)"),open(os.path.join(META,"ambiente.json"),"w"),indent=2)
print("AVALIACAO COMPLETA",flush=True)


In [ ]:
!python3 -u src/08_avaliacao.py && cat results/tables/tab03_metricas_globais.csv

### 12. Registro de figuras e tabelas
Copia os artefatos relevantes para `apresentacao/` e gera o Excel de referenciamento (ID, arquivo, legenda, descrição, explicação breve, seção do artigo, insumo do WP1A).

In [ ]:
%%writefile /content/ProjetoA_WP1A/src/09_registro.py
"""09_registro.py — copia figuras/tabelas/dados relevantes para ../apresentacao/ e gera o Excel de referenciamento
(ID, tipo, arquivo, título/legenda, descrição, explicação breve, fonte, seção do artigo, insumo §12 do WP1A)."""
import os, shutil, glob, json, pandas as pd
ROOT=os.environ.get("WP1A_ROOT") or os.path.join(os.path.dirname(os.path.abspath(__file__)),".."); AP=os.path.join(ROOT,"..","apresentacao")
TAB=os.path.join(ROOT,"results","tables"); FIG=os.path.join(ROOT,"results","figures"); META=os.path.join(ROOT,"results","metadata")
for d in ("figuras","tabelas","dados"): os.makedirs(os.path.join(AP,d),exist_ok=True)
REG=[ # (arquivo, tipo, título/legenda, descrição, explicação breve p/ leigo, seção do artigo, insumo §12)
 ("tab01_caracterizacao_base.csv","tabela","Tabela 1 – Caracterização da base de dados","Linhas, colunas, classes, runs por classe, amostras por run, nulos, infinitos e duplicatas de cada um dos 4 arquivos do dataset Rieth et al. (2017).","É o inventário: diz exatamente o que há na base antes de qualquer modelo. Sem isso, um erro nos dados vira erro no resultado sem ninguém perceber.","II. Metodologia (2.1)","1"),
 ("manifesto_divisao.csv","dado","Manifesto de divisão por execução (run)","Uma linha por (classe, run) com o conjunto ao qual pertence: treino (100/classe), validação (50/classe), teste (500/classe), não usado.","É a prova de que nenhuma simulação foi usada para ensinar e testar ao mesmo tempo — qualquer pessoa pode conferir.","II. Metodologia (2.2)","A3"),
 ("tab_piloto_vazamento.csv","tabela","Tabela – Efeito da unidade de divisão (piloto)","Mesmo modelo (árvore), mesmo volume de dados: divisão por run vs. divisão aleatória por amostra. Acurácia, acurácia balanceada, F1 macro e MCC.","Mede o vazamento nos nossos próprios dados. Se a versão embaralhada acerta muito mais, a diferença não é desempenho — é o modelo reconhecendo vizinhos que já viu.","III. Resultados (3.1)","8"),
 ("fig07_vazamento.png","figura","Figura – Efeito do vazamento de dados","Barras comparando as quatro métricas nas duas formas de divisão.","Visualização do experimento acima.","III. Resultados (3.1)","8"),
 ("fig_eda_pca.png","figura","Figura – Projeção PCA das 21 classes","Duas primeiras componentes principais, 2.000 amostras por classe (treino), normal em preto.","Comprime 52 medições em duas dimensões desenháveis. Classes que se sobrepõem na figura são as que os modelos vão confundir.","III. Resultados (3.1)","A2"),
 ("fig_eda_correlacao.png","figura","Figura – Correlação entre as 52 variáveis","Matriz de correlação de Pearson (treino, padronizado).","Mostra quais sensores 'andam juntos'. Muita correlação significa redundância — informação repetida.","II. Metodologia / apêndice","A2"),
 ("fig_eda_separacao.png","figura","Figura – Afastamento de cada falha em relação ao normal","Por variável e por falha: |média_falha − média_normal| / desvio_normal.","Quanto mais escura a linha de uma falha, mais parecida com a operação normal — e mais difícil de detectar.","III. Resultados (3.1)","A2"),
 ("tab02_hiperparametros.csv","tabela","Tabela 2 – Hiperparâmetros selecionados","Configuração final de cada modelo, escolhida pelo F1 macro na validação (nunca no teste).","Cada modelo tem 'botões' de ajuste. Estes são os valores escolhidos — usando só a validação, para não contaminar o teste.","II. Metodologia (2.4)","2"),
 ("tab09_busca_hp_resumo.csv","tabela","Tabela – Resumo da busca de hiperparâmetros","Número de configurações testadas por modelo e faixa de F1 macro na validação.","Mostra quanto cada modelo variou conforme o ajuste — modelos sensíveis ao ajuste exigem mais cuidado.","II. Metodologia (2.4)","2"),
 ("tab09b_nystroem_tradeoff.csv","tabela","Tabela – SVM via Nyström: qualidade × custo","F1 macro, MCC e tempo em função do número de componentes D e de C.","O SVM exato é inviável nesta escala; a aproximação tem um 'botão' (D) que troca tempo por qualidade. A tabela mostra o preço de cada escolha.","III. Resultados (3.4)","7"),
 ("fig06_svm_nystroem_tradeoff.png","figura","Figura – SVM via Nyström em função de D","Curvas de F1 macro e tempo de treino vs. D.","Visualização do compromisso acima.","III. Resultados (3.4)","7"),
 ("tab03_metricas_globais.csv","tabela","Tabela 3 – Métricas globais no teste","Média, desvio-padrão e IC 95% (5 sementes) de acurácia, acurácia balanceada, F1 macro, F1 ponderada e MCC; também com rótulo do run.","O resultado central. Cinco repetições e intervalo de confiança: é a diferença entre 'deu 87%' e 'dá entre 86,5% e 87,5%'.","III. Resultados (3.2)","3"),
 ("fig01_f1_mcc_comparativo.png","figura","Figura 1 – F1 macro e MCC por modelo","Barras com IC 95%.","F1 macro dá o mesmo peso a cada classe; MCC só é alto quando toda a matriz de acertos está boa. Juntos, evitam que uma acurácia alta esconda classes ignoradas.","III. Resultados (3.2)","6"),
 ("tab04_metricas_por_classe.csv","tabela","Tabela 4 – Precisão, revocação e F1 por classe","Por modelo e por classe (média de 5 sementes).","Mostra onde cada modelo acerta e onde falha, falha por falha.","III. Resultados (3.3)","4"),
 ("fig04_f1_por_classe_heatmap.png","figura","Figura – F1 por classe (mapa de calor)","Modelos × 21 classes.","Colunas escuras em todos os modelos = falhas intrinsecamente difíceis, não fraqueza de um algoritmo.","III. Resultados (3.3)","4"),
 ("tab06_falhas_confundidas.csv","tabela","Tabela – Pares de classes mais confundidos","Para cada modelo, os 10 pares (verdadeira → predita) com maior fração fora da diagonal.","Diz *com o que* o modelo confunde cada falha — informação de engenharia, não só estatística.","III. Resultados (3.3)","8"),
 ("tab06b_dificuldade_por_classe.csv","tabela","Tabela – Dificuldade por classe","F1 médio entre modelos, por classe, ordenado.","Ranking das falhas mais difíceis, independente do algoritmo.","III. Resultados (3.3)","8"),
 ("tab05_custo_computacional.csv","tabela","Tabela 5 – Custo computacional","Tempo de treino, tempo de inferência (total e por amostra), tamanho do modelo e pico de memória — mesmo hardware, 5 sementes.","Um modelo 1% melhor mas 50× mais lento não é 'melhor', é diferente. Em planta industrial a resposta precisa vir em segundos.","III. Resultados (3.4)","5"),
 ("fig02_custo_computacional.png","figura","Figura 2 – Custo computacional","Três painéis em escala log: treino, inferência por amostra, tamanho.","Visualização da tabela acima.","III. Resultados (3.4)","7"),
 ("fig05_tradeoff_f1_vs_inferencia.png","figura","Figura – Desempenho × custo de inferência","Dispersão F1 macro vs. tempo por amostra (log).","O 'melhor compromisso' (QP5 do WP1A) está no canto superior esquerdo: alto F1, baixo tempo.","III. Resultados (3.4)","7"),
 ("tab07_estatistica.csv","tabela","Tabela – Comparação estatística","Friedman (global) e Wilcoxon pareado com correção de Holm, sobre F1 macro e MCC nas 5 sementes.","Responde se as diferenças entre modelos são maiores que a variação por acaso entre repetições.","III. Resultados (3.2)","3"),
 ("tab08_run_level.csv","tabela","Tabela – Métricas em nível de execução (voto majoritário)","F1 macro, acurácia e MCC agregando as predições de cada run por voto majoritário.","Em vez de julgar cada leitura de 3 min, julga a simulação inteira. É como Koçak et al. (2026) reportam — permite comparação direta.","III. Resultados (3.5)","3"),
 ("ambiente.json","dado","Metadados do ambiente computacional","Versões de Python e bibliotecas, hardware, sementes.","Permite a outra pessoa reproduzir exatamente o que foi feito.","II. Metodologia (2.5)","10"),
 ("busca_hp_resultados.json","dado","Resultados brutos da busca de hiperparâmetros","Todas as configurações testadas com F1 macro, MCC e tempo (validação).","Registro completo do ajuste — nada foi escolhido 'de cabeça'.","apêndice / repositório","9"),
 ("auditoria.json","dado","Auditoria completa (JSON)","Todos os itens do §9.1 do WP1A por arquivo.","Versão detalhada da Tabela 1.","repositório","1"),
]
for m in ["regressao_logistica","arvore_decisao","random_forest","gradient_boosting","xgboost","svm_nystroem"]:
    REG.append((f"fig03_matriz_confusao_{m}.png","figura",f"Figura – Matriz de confusão: {m}","Normalizada por linha, 5 sementes agregadas, rótulo físico.","Cada linha é a classe verdadeira; cada coluna, o que o modelo disse. A diagonal são os acertos; o resto, as confusões.","III. Resultados (3.3) / apêndice","5"))
    REG.append((f"tab_matriz_confusao_{m}.csv","tabela",f"Tabela – Matriz de confusão (CSV): {m}","Valores normalizados da figura correspondente.","Versão numérica da matriz.","repositório","5"))
rows=[]
for i,(arq,tipo,tit,desc,expl,sec,ins) in enumerate(REG,1):
    src=None
    for base in (TAB,FIG,META):
        if os.path.exists(os.path.join(base,arq)): src=os.path.join(base,arq); break
    dst_dir={"figura":"figuras","tabela":"tabelas","dado":"dados"}[tipo]; status="ok" if src else "AUSENTE"
    if src: shutil.copy2(src,os.path.join(AP,dst_dir,arq))
    rows.append(dict(ID=f"{tipo[0].upper()}{i:02d}",Tipo=tipo,Arquivo=f"{dst_dir}/{arq}",Titulo_Legenda=tit,Descricao=desc,Explicacao_breve=expl,Secao_do_artigo=sec,Insumo_WP1A_s12=ins,Fonte="Autores (2026), a partir de Rieth et al. (2017)",Status=status))
df=pd.DataFrame(rows); xl=os.path.join(AP,"REGISTRO-FIGURAS-TABELAS.xlsx")
with pd.ExcelWriter(xl,engine="openpyxl") as w:
    df.to_excel(w,sheet_name="Registro",index=False); ws=w.sheets["Registro"]
    for col,wd in zip("ABCDEFGHIJ",(6,8,42,48,70,80,26,10,34,9)): ws.column_dimensions[col].width=wd
df.to_csv(os.path.join(AP,"REGISTRO-FIGURAS-TABELAS.csv"),index=False)
print(df[["ID","Arquivo","Status"]].to_string(index=False)); print(f"\n{(df.Status=='ok').sum()}/{len(df)} presentes → {xl}")


In [ ]:
!python3 -u src/09_registro.py